In [49]:
import pandas as pd
import numpy as np

import joblib

from scipy.stats import poisson

In [35]:
model = joblib.load(
    '../models/xgb_wc_predictor.pkl'
)

In [36]:
team_profiles = pd.read_csv(
    '../data/cleaned/team_strength_profiles.csv'
)

In [37]:
team_stats_dict = (

    team_profiles

    .set_index('team')

    .to_dict(orient='index')
)

In [38]:
elo = pd.read_csv(
    '../data/raw/elo_ratings.csv'
)

elo['snapshot_date'] = pd.to_datetime(
    elo['snapshot_date']
)

In [39]:
latest_elo = (

    elo.sort_values('snapshot_date')

       .groupby('country')

       .tail(1)
)

In [40]:
elo_dict = dict(

    zip(

        latest_elo['country'],
        latest_elo['rating']
    )
)

In [41]:
def create_match_features(
    home_team,
    away_team
):

    # --------------------------------------------------------
    # ELO
    # --------------------------------------------------------

    home_elo = elo_dict.get(
        home_team,
        1500
    )

    away_elo = elo_dict.get(
        away_team,
        1500
    )



    # --------------------------------------------------------
    # TEAM STATS
    # --------------------------------------------------------

    home_stats = team_stats_dict.get(
        home_team,
        {}
    )

    away_stats = team_stats_dict.get(
        away_team,
        {}
    )



    # --------------------------------------------------------
    # ATTACK
    # --------------------------------------------------------

    home_attack = home_stats.get(
        'avg_goals_scored',
        1.2
    )

    away_attack = away_stats.get(
        'avg_goals_scored',
        1.2
    )



    # --------------------------------------------------------
    # DEFENSE
    # --------------------------------------------------------

    home_defense = home_stats.get(
        'avg_goals_conceded',
        1.2
    )

    away_defense = away_stats.get(
        'avg_goals_conceded',
        1.2
    )



    # --------------------------------------------------------
    # WIN RATE
    # --------------------------------------------------------

    home_win_pct = home_stats.get(
        'win_pct',
        0.5
    )

    away_win_pct = away_stats.get(
        'win_pct',
        0.5
    )



    # --------------------------------------------------------
    # ELITE PERFORMANCE
    # --------------------------------------------------------

    home_elite = home_stats.get(
        'elite_wins',
        0
    )

    away_elite = away_stats.get(
        'elite_wins',
        0
    )



    # --------------------------------------------------------
    # WORLD CUP UPSETS
    # --------------------------------------------------------

    home_upsets = home_stats.get(
        'world_cup_upsets',
        0
    )

    away_upsets = away_stats.get(
        'world_cup_upsets',
        0
    )



    # --------------------------------------------------------
    # ADJUSTED STRENGTH
    # --------------------------------------------------------

    home_strength = home_stats.get(
        'adjusted_strength',
        100
    )

    away_strength = away_stats.get(
        'adjusted_strength',
        100
    )



    # --------------------------------------------------------
    # BUILD FEATURE VECTOR
    # --------------------------------------------------------

    features = pd.DataFrame({

        'home_elo': [home_elo],

        'away_elo': [away_elo],

        'elo_diff': [
            home_elo - away_elo
        ],

        'home_rank': [50],

        'away_rank': [50],

        'rank_diff': [0],

        'neutral': [1],

        'is_competitive': [1],

        'home_form': [
            home_win_pct * 3
        ],

        'away_form': [
            away_win_pct * 3
        ],

        'form_diff': [
            home_win_pct - away_win_pct
        ],

        'home_attack_strength': [
            home_attack
        ],

        'away_attack_strength': [
            away_attack
        ],

        'home_defense_strength': [
            home_defense
        ],

        'away_defense_strength': [
            away_defense
        ],

        'elite_score_diff': [
            home_elite - away_elite
        ],

        'adjusted_strength_diff': [
            home_strength - away_strength
        ]
    })

    return features

In [42]:
create_match_features(
    'France',
    'Argentina'
)

,home_elo,away_elo,elo_diff,home_rank,away_rank,rank_diff,neutral,is_competitive,home_form,away_form,form_diff,home_attack_strength,away_attack_strength,home_defense_strength,away_defense_strength,elite_score_diff,adjusted_strength_diff
0,2081,2113,-32,50,50,0,1,1,1.918919,2.037736,-0.039606,2.081081,1.990566,0.855856,0.603774,15,50.571296


In [43]:
def predict_match(
    home_team,
    away_team
):

    features = create_match_features(
        home_team,
        away_team
    )

    probs = model.predict_proba(
        features
    )[0]

    return probs

In [44]:
predict_match(
    'France',
    'Argentina'
)

array([0.45788226, 0.39627898, 0.14583878], dtype=float32)

In [50]:
def expected_goals(
    home_team,
    away_team
):

    home_stats = team_stats_dict.get(
        home_team,
        {}
    )

    away_stats = team_stats_dict.get(
        away_team,
        {}
    )



    # --------------------------------------------------------
    # ATTACK STRENGTH
    # --------------------------------------------------------

    home_attack = home_stats.get(
        'avg_goals_scored',
        1.5
    )

    away_attack = away_stats.get(
        'avg_goals_scored',
        1.2
    )



    # --------------------------------------------------------
    # DEFENSE
    # LOWER = BETTER
    # --------------------------------------------------------

    home_defense = home_stats.get(
        'avg_goals_conceded',
        1.0
    )

    away_defense = away_stats.get(
        'avg_goals_conceded',
        1.0
    )



    # --------------------------------------------------------
    # ADJUSTED STRENGTH
    # --------------------------------------------------------

    home_strength = home_stats.get(
        'adjusted_strength',
        100
    )

    away_strength = away_stats.get(
        'adjusted_strength',
        100
    )



    # --------------------------------------------------------
    # ELO
    # --------------------------------------------------------

    home_elo = elo_dict.get(
        home_team,
        1500
    )

    away_elo = elo_dict.get(
        away_team,
        1500
    )



    # --------------------------------------------------------
    # ELO FACTOR
    # --------------------------------------------------------

    elo_factor = (

        (home_elo - away_elo)

        / 400
    )



    # --------------------------------------------------------
    # EXPECTED GOALS
    # --------------------------------------------------------

    home_xg = (

        (
            home_attack
            *
            away_defense
        )

        *

        (
            home_strength
            / away_strength
        )

        *

        (
            1 + elo_factor
        )
    )



    away_xg = (

        (
            away_attack
            *
            home_defense
        )

        *

        (
            away_strength
            / home_strength
        )

        *

        (
            1 - elo_factor
        )
    )



    # --------------------------------------------------------
    # SAFETY LIMITS
    # --------------------------------------------------------

    home_xg = max(
        0.2,
        min(home_xg, 4)
    )

    away_xg = max(
        0.2,
        min(away_xg, 4)
    )



    return home_xg, away_xg

In [51]:
def simulate_score(
    home_team,
    away_team
):

    home_xg, away_xg = expected_goals(
        home_team,
        away_team
    )



    home_goals = poisson.rvs(
        home_xg
    )



    away_goals = poisson.rvs(
        away_xg
    )



    return home_goals, away_goals

In [52]:
simulate_score(
    'France',
    'Argentina'
)

(2, 1)

In [48]:
for i in range(10):

    print(
        simulate_score(
            'France',
            'Argentina'
        )
    )

(1, 2)
(0, 6)
(1, 3)
(1, 2)
(1, 2)
(1, 3)
(1, 5)
(0, 0)
(0, 3)
(2, 0)
